# Train Distractor Generation Model — Llama-3.2-3B + QLoRA on RACE (DG-RACE format)

**Environment:** Kaggle T4x2 (2x NVIDIA Tesla T4, 16GB each)

**Task:** Fine-tune Llama to generate 3 distractors given (context, question, correct_answer)

**Dataset:** RACE (Reading Comprehension from Examinations) converted to DG-RACE format

In [1]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("unsloth")
install("trl>=0.8.6")
install("peft>=0.10.0")
install("bitsandbytes>=0.43.0")
install("accelerate>=0.27.0")
install("datasets>=2.14.0")
install("transformers>=4.40.0")
install("sentencepiece")
install("protobuf")

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

All packages installed.


In [2]:
import os
import json
import random
import torch
from kaggle_secrets import UserSecretsClient

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Get HuggingFace token from Kaggle Secrets
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found. Add it to Kaggle Secrets.")

CONFIG = {
    "base_model": "meta-llama/Llama-3.2-3B-Instruct",
    "output_dir": "/kaggle/working/distractor_model_output",
    "hf_repo": "",          # Set to "username/llama-distractor-race" to push to Hub
    "race_subset": "all",   # "all" = high + middle school combined
    "max_train_samples": 10000,  
    "max_val_samples": 1000,     
    "max_seq_length": 512,       
    # Training
    "num_epochs": 2,             
    "per_device_train_batch_size": 16,  
    "per_device_eval_batch_size": 8,    
    "gradient_accumulation_steps": 1,   
    "learning_rate": 2e-4,
    "weight_decay": 0.0,        
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",
    "logging_steps": 100,
    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

HF_TOKEN loaded from Kaggle Secrets.


In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,           # Auto-detect
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
USING_UNSLOTH = True
print("Using Unsloth for accelerated training.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # For training

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Using Unsloth for accelerated training.
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [4]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are an expert at creating plausible but incorrect answer choices "
    "(distractors) for multiple choice questions."
)

ANSWER_MAP = {"A": 0, "B": 1, "C": 2, "D": 3}


def convert_race_to_dgrace(sample: dict) -> dict:
    """Convert RACE sample to DG-RACE training format."""
    try:
        answer_letter = sample["answer"].strip().upper()
        if answer_letter not in ANSWER_MAP:
            return {"text": "", "valid": False}

        correct_idx = ANSWER_MAP[answer_letter]
        options = sample["options"]

        if len(options) != 4:
            return {"text": "", "valid": False}

        correct_answer = options[correct_idx]
        distractors = [opt for i, opt in enumerate(options) if i != correct_idx]

        if len(distractors) != 3 or not correct_answer.strip():
            return {"text": "", "valid": False}

        if any(not d.strip() for d in distractors):
            return {"text": "", "valid": False}

        # Shuffle distractor order
        random.shuffle(distractors)

        # Truncate article if too long
        article = sample["article"]
        article_words = article.split()
        if len(article_words) > 400:
            article = " ".join(article_words[:400])

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    f"Passage: {article}\n"
                    f"Question: {sample['question']}\n"
                    f"Correct Answer: {correct_answer}\n\n"
                    "Generate exactly 3 plausible but incorrect answer choices."
                ),
            },
            {
                "role": "assistant",
                "content": (
                    f"Distractor 1: {distractors[0]}\n"
                    f"Distractor 2: {distractors[1]}\n"
                    f"Distractor 3: {distractors[2]}"
                ),
            },
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        return {"text": text, "valid": True}

    except (KeyError, IndexError, TypeError):
        return {"text": "", "valid": False}


print("Loading RACE dataset...")
train_raw = load_dataset("ehovy/race", CONFIG["race_subset"], split="train", trust_remote_code=True)
val_raw = load_dataset("ehovy/race", CONFIG["race_subset"], split="validation", trust_remote_code=True)
print(f"Train: {len(train_raw):,} samples")
print(f"Val:   {len(val_raw):,} samples")

# Subsample
if CONFIG["max_train_samples"] and len(train_raw) > CONFIG["max_train_samples"]:
    train_raw = train_raw.shuffle(seed=42).select(range(CONFIG["max_train_samples"]))
if CONFIG["max_val_samples"] and len(val_raw) > CONFIG["max_val_samples"]:
    val_raw = val_raw.shuffle(seed=42).select(range(CONFIG["max_val_samples"]))

# Convert to DG-RACE format (dùng num_proc để tăng tốc)
train_dataset = train_raw.map(
    convert_race_to_dgrace,
    remove_columns=train_raw.column_names,
    num_proc=4,
    desc="Converting train to DG-RACE",
)
val_dataset = val_raw.map(
    convert_race_to_dgrace,
    remove_columns=val_raw.column_names,
    num_proc=4,
    desc="Converting val to DG-RACE",
)

train_dataset = train_dataset.filter(lambda x: x["valid"]).remove_columns(["valid"])
val_dataset = val_dataset.filter(lambda x: x["valid"]).remove_columns(["valid"])

print(f"Final train: {len(train_dataset):,}")
print(f"Final val:   {len(val_dataset):,}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ehovy/race' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
[datasets.load|ERROR]`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ehovy/race' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading RACE dataset...


README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

...

Converting train to DG-RACE (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

Converting val to DG-RACE (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Final train: 10,000
Final val:   1,000


In [5]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    bf16=False,
    fp16=True,
    optim="paged_adamw_32bit",
    gradient_checkpointing="unsloth",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=CONFIG["logging_steps"],
    report_to="none",
    save_total_limit=1,         
    load_best_model_at_end=True, 
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,              
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataset_num_proc=4,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [6]:
import time

print("Starting distractor generation training...")
start_time = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed / 3600:.2f} hours.")
print(f"Train loss: {trainer_stats.training_loss:.4f}")

Starting distractor generation training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,652 | Num Epochs = 2 | Total steps = 1,208
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
500,1.915121,1.900487
1000,1.793436,1.899238
1208,1.802424,1.898360


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/distractor_model_output/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/distractor_model_output/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/distractor_model_output/checkpoint-1208/tokenizer_config.json.



Training completed in 3.94 hours.
Train loss: 1.8907


In [7]:
from huggingface_hub import login

# Save locally
os.makedirs(CONFIG["output_dir"], exist_ok=True)
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Distractor model saved to: {CONFIG['output_dir']}")

# Save merged model (optional — larger but faster for inference)
MERGED_DIR = CONFIG["output_dir"] + "_merged"
try:
    if USING_UNSLOTH:
        model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    else:
        merged = model.merge_and_unload()
        merged.save_pretrained(MERGED_DIR)
        tokenizer.save_pretrained(MERGED_DIR)
    print(f"Merged model saved to: {MERGED_DIR}")
except Exception as e:
    print(f"Warning: Could not save merged model: {e}")

# Push adapter to HuggingFace Hub
# if CONFIG["hf_repo"]:
#     login(token=HF_TOKEN)
#     model.push_to_hub(CONFIG["hf_repo"])
#     tokenizer.push_to_hub(CONFIG["hf_repo"])
#     print(f"Pushed adapter to: https://huggingface.co/{CONFIG['hf_repo']}")
# else:
#     print("HF_REPO not set — skipping Hub push.")
#     print("Download from Kaggle Output: /kaggle/working/distractor_model_output")

# List output files
print("\nOutput files:")
for root, dirs, files in os.walk(CONFIG["output_dir"]):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {path} ({size:.1f} MB)")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/distractor_model_output/tokenizer_config.json.


Distractor model saved to: /kaggle/working/distractor_model_output


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/distractor_model_output_merged/tokenizer_config.json.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:17<00:00,  8.60s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:47<00:00, 23.97s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/distractor_model_output_merged`
Merged model saved to: /kaggle/working/distractor_model_output_merged

Output files:
  /kaggle/working/distractor_model_output/tokenizer_config.json (0.0 MB)
  /kaggle/working/distractor_model_output/README.md (0.0 MB)
  /kaggle/working/distractor_model_output/tokenizer.json (16.4 MB)
  /kaggle/working/distractor_model_output/chat_template.jinja (0.0 MB)
  /kaggle/working/distractor_model_output/adapter_config.json (0.0 MB)
  /kaggle/working/distractor_model_output/adapter_model.safetensors (92.8 MB)
  /kaggle/working/distractor_model_output/checkpoint-1208/rng_state.pth (0.0 MB)
  /kaggle/working/distractor_model_output/checkpoint-1208/optimizer.pt (185.7 MB)
  /kaggle/working/distractor_model_output/checkpoint-1208/scaler.pt (0.0 MB)
  /kaggle/working/distractor_model_output/checkpoint-1208/tokenizer_config.json (0.0 MB)
  /kaggle/working/distractor_model_output/checkpoint-1208/README.md (0.0 M